In [ ]:
#| default_exp clients

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import networkx as nx
import pickle
import json
from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from peft import *
from community import community_louvain
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from transformers import AutoTokenizer
from omegaconf.dictconfig import DictConfig
import numpy as np
import math
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
#| export
class SFMTL(BaseClient):
    def __init__(self, 
                 id,
                 cfg,
                 state= None,
                 role= AgentRole.CLIENT,
                 block= None):
        
        super().__init__(id, cfg, state, role, block)
        
        if self.role == AgentRole.CLIENT:
            self.anchorloss = AnchorLoss(self.cfg.random_seed, self.cfg.data.num_classes, self.cfg.model.hidden_dim, self.t, self.h_c).to(self.device)
            self.label_set = list(set(np.array([batch['y'] for batch in self.train_ds])))

In [ ]:
#| export 
@patch
def server_init(self: SFMTL, client_fn, client_selector, client_cls, loss_fn, writer):
    BaseClient.server_init(self, client_fn, client_selector, client_cls, loss_fn, writer)
    self.classes = self.cfg.data.classes
    self.idx_to_cls = {i: self.classes[i] for i in range(len(self.classes))}

### Training

In [ ]:
#| export
@patch
def _forward(self: SFMTL, batch):
    X, y = batch['x'], batch['y']
    y_copied = deepcopy(y)
    labels = y.type(torch.LongTensor).to(self.device)
    ys = labels.float()
    
    h = self.model.encoder(X)
    outputs = self.model.classifier(h)    
    
    loss_anchor = self.anchorloss(h, ys, Lambda = self.cfg.lambda_anchor)
    loss = self.criterion(outputs, y_copied)
    
    return loss + loss_anchor, h, outputs, labels

In [ ]:
#| export
@patch
def _closure(self: SFMTL, batch: dict) -> tuple:
    metrics = {k: 0 for k in list(self.cfg.training_metrics)}  # Ensure metrics is always defined
    h, labels = None, None
    try:
        loss, h, logits, labels = self._forward(batch)

        probs = torch.nn.functional.softmax(logits, dim=-1)
        y_pred = probs.argmax(dim=-1)
        y_true = batch[self.label_key]

        if hasattr(self, "training_metrics") and self.cfg.training_metrics:
            if hasattr(self, "tokenizer"):
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true, tokenizer=self.tokenizer)
            else:
                metrics = self.training_metrics.compute(y_pred=y_pred, y_true=y_true)

    except Exception as e:
        print(e)
        return torch.tensor(0.0, dtype=torch.float32, device=self.device), metrics, h, labels  # Return safe values

    return loss, metrics, h, labels


In [ ]:
#| export
@patch
def _run_batch(self: SFMTL, batch: dict) -> tuple:
    batch_mean_anchor = torch.zeros(self.cfg.data.num_classes, self.cfg.model.hidden_dim).to(self.device)
    self.optimizer.zero_grad()

    loss, metrics, h, labels = self._closure(batch)
    if loss.item() == 0.0 or h is None:
        return loss, metrics, batch_mean_anchor
    
    for i in set(labels.tolist()):
        batch_mean_anchor[i] += torch.mean(h[labels==i],dim=0)
   
    loss.backward()
    
    if self.cfg.model.grad_norm_clip:
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.model.grad_norm_clip)

    self.optimizer.step()

    return loss, metrics, batch_mean_anchor

In [ ]:
#| export
@patch
def _run_epoch(self: SFMTL):

    epoch_mean_anchor = torch.zeros(self.cfg.data.num_classes, self.cfg.model.hidden_dim).to(self.device)
    with torch.no_grad():
        epoch_mean_anchor.copy_(self.anchorloss.anchor)

    for batch_idx, batch in enumerate(self.train_loader):
        batch = self.send_to_device(batch)
        _, _, batch_mean_anchor = self._run_batch(batch)

        for i in self.label_set:
            #compute batch mean anchor according to batch label
            batch_mean_anchor[i] = batch_mean_anchor[i]/(batch_idx+1)

            #compute epoch mean anchor according to batch mean anchor
            lambda_momentum = self.cfg.momentum_anchor #pow(2, -(epoch+1))
            # epoch_mean_anchor[i] = lambda_momentum * epoch_mean_anchor[i] + (1-lambda_momentum)*batch_mean_anchor[i]
            epoch_mean_anchor[i].mul_(lambda_momentum).add_((1 - lambda_momentum) * batch_mean_anchor[i])

        

    # self.anchorloss.anchor =  torch.nn.Parameter(epoch_mean_anchor, requires_grad=False)
    with torch.no_grad():
        self.anchorloss.anchor.copy_(epoch_mean_anchor)


### Communication

The communication in DMTL is a matter of sending (saving to the disk) two things, the classification head and a randomly picked data represntation.

In [ ]:
#| export
@patch
def save_state(self: SFMTL, state_dict):  # noqa: F811
    # save the model to self.cfg.save_dir/comm_round/f"local_output_{id}"/state.pth
    
    
    state_dict['model'] = self.model.state_dict()
    state_dict['optimizer'] = self.optimizer.state_dict()
    state_dict['h'] = self.anchorloss.anchor.detach().clone() # (num_classes, hidden_size)
    state_dict['label_set'] = self.label_set
    
    state_path = os.path.join(self.cfg.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
    if not os.path.exists(os.path.dirname(state_path)):
        os.makedirs(os.path.dirname(state_path))

    torch.save(state_dict, state_path)

    if self.role == AgentRole.CLIENT:
        save_space(self)


### Server side

At the server, we are doing the followin steps:
- Form the coalitions.
  - First, construct the weighted undirected graph $\mathcal{g}$.
  - Second, pass this graph to the louvian algorithm to get the communities.
- aggregate based on the equations stated in the next cells.


First, model similrity is compared using the norm of the difference between the two models as follows:
$$\operatorname{Sim}_{\text {head }}(\bm{w}_k,\bm{w}_\ell)=\left\|\bm{w}_k-\bm{w}_\ell\right\|,$$


In [ ]:
#| export
@patch
def model_similarity(self: SFMTL, h1, h2, model1, model2):

    model_cls = get_cls("fedai.vision.models", self.cfg.model.name)

    m1 = model_cls().classifier
    m2 = model_cls().classifier

    with torch.no_grad():
        for k, v in model1.items():
            if k.startswith("classifier."):
                sub_k = k[len("classifier."):]
                if sub_k in m1.state_dict():
                    print(f"copying {sub_k} to m1")
                    m1.state_dict()[sub_k].copy_(v)
                else:
                    print(f"Key {sub_k} not found in m1.state_dict()")

        for k, v in model2.items():
            if k.startswith("classifier."):
                sub_k = k[len("classifier."):]
                if sub_k in m2.state_dict():
                    print(f"copying {sub_k} to m2")
                    m2.state_dict()[sub_k].copy_(v)
                else:
                    print(f"Key {sub_k} not found in m1.state_dict()")


    m1.to(self.device)
    m2.to(self.device)
    
    avg_sim = 0.0
    for h in [h1, h2]:
        h = h.to(self.device)
        with torch.no_grad():
            out1 = m1(h)
            out2 = m2(h)
            sim = F.cosine_similarity(out1, out2, dim=1).mean()
            avg_sim += sim.item()

    return avg_sim / 2  # correct normalization


We compare the embedding closeness using cosine similiairy.


$$\operatorname{Sim}_{\mathrm{repr}}(k, \ell) = \cos(\Omega) = \frac{\mathbf{h}_k \cdot \mathbf{h}_{\ell}}{\|\mathbf{h}_k\| \|\mathbf{h}_{\ell}\|}$$

In [ ]:
#| export
@patch
def h_similarity(self: SFMTL, h1, h2, label_set, label_set2):
    h1 = h1.reshape(self.cfg.data.num_classes, self.cfg.model.hidden_dim)
    h2 = h2.reshape(self.cfg.data.num_classes, self.cfg.model.hidden_dim)
    
    h1_norm = F.normalize(h1, p=2, dim=1)  # (3, 512)
    h2_norm = F.normalize(h2, p=2, dim=1)  # (3, 512)
    
    cos_sim_matrix = torch.mm(h1_norm, h2_norm.T)  # (10, 10)
    max_similarity = cos_sim_matrix.max(dim=1).values.mean()# This gives a higher similarity score if each class in h1 has at least one good match in h2.
    data = cos_sim_matrix.cpu().numpy()

    # index only data for the label_set
    data = data[label_set][:, label_set2]

    cols1 = [self.idx_to_cls[i] for i in label_set]
    cols2 = [self.idx_to_cls[i] for i in label_set2]

    df = pd.DataFrame(data, columns= cols1, index= cols2)
    return df, max_similarity.item()

We build a similarity graph using both the local represntation $h$ and the classification head $w$ where similairty is defined as:
$$\operatorname{Sim} = \sum_{i \in S} \alpha \cdot \operatorname{Sim}_{\text {head }}(C_j)+(1-\alpha) \cdot \operatorname{Sim}_{\mathrm{repr}}(C_j)$$

In [ ]:
#| export
@patch
def build_graph(self: SFMTL, lst_active_ids, comm_round):

    num_active = len(lst_active_ids)
    graph = np.zeros((num_active, num_active))

    clients_sim_dict = {}
    visited = {}
    for i, id in enumerate(lst_active_ids):
        state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
        state = torch.load(state_path, weights_only= False)
        model1 = state['model']
        h1 = state['h']
        label_set = state['label_set']

        for j, other_id in enumerate(lst_active_ids):
            if i == j or (id, other_id) in visited:
                continue
            other_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{other_id}", "state.pth")
            other_state = torch.load(other_state_path, weights_only= False)
            model2 = other_state['model']
            h2 = other_state['h']
            label_set2 = other_state['label_set']

            w_sim = self.model_similarity(h1, h2, model1, model2)
            h_sim_df, h_sim = self.h_similarity(h1, h2, label_set, label_set2)
            clients_sim_dict[(id, other_id)] = h_sim_df
            
            val = (self.cfg.alpha) * w_sim + (1 - self.cfg.alpha) * h_sim
            print(f"Client {id} and {other_id} similarity: {val}")
            print(f"Client {id} and {other_id} h similarity: {h_sim}")
            print(f"Client {id} and {other_id} w similarity: {w_sim}")
            val = round(val, 3)
            val = max(val, 0)  # prevent negative edge weights
            graph[i][j] = graph[j][i] = val

            visited[(id, other_id)] = True
            visited[(other_id, id)] = True

    print("Before sym:", graph)
    


    edges = []
    for i in range(num_active):
        for j in range(num_active):
            if i != j:
                edges.append((i, j, graph[i][j]))
                
    G = nx.Graph()
    G.add_weighted_edges_from(edges)

    for node, label in zip(list(range(num_active)), lst_active_ids):
        G.nodes[node]['label'] = label
    
    df_path = os.path.join(self.cfg.save_dir, str(comm_round), f"h_sim_df_{str(comm_round)}.pth")
    if not os.path.exists(os.path.dirname(df_path)):
        os.makedirs(os.path.dirname(df_path))
    torch.save(clients_sim_dict, df_path)

    return G, graph

In [ ]:
#| export
@patch    # save the model to self.cfg.save_dir/comm_round/f"local_output_{id}"/state.pth

def build_random_graph(self: SFMTL, lst_active_ids, comm_round):

    num_active = len(lst_active_ids)

    # seed the random number generator for reproducibility
    np.random.seed(42)
    clients_sim_dict = {}
    visited = {}


    G = nx.erdos_renyi_graph(num_active, 0.3)
    for (u, v) in G.edges():
        G.edges[u, v]['weight'] = round(random.uniform(0, 1), 3)

    zero_mat = np.zeros((num_active, num_active))

    for i in range(num_active):
        for j in range(num_active):
            if i != j:
                # checj if i and j are connected
                if G.has_edge(i, j):
                    zero_mat[i][j] = G.edges[i, j]['weight']
                

    for node, label in zip(list(range(num_active)), lst_active_ids):
        G.nodes[node]['label'] = label
    
    df_path = os.path.join(self.cfg.save_dir, str(comm_round), f"h_sim_df_{str(comm_round)}.pth")
    if not os.path.exists(os.path.dirname(df_path)):
        os.makedirs(os.path.dirname(df_path))
    torch.save(clients_sim_dict, df_path)

    return G, zero_mat

Now, we have a wighted graph, we nedd to form the coalitions (detecting the communities). We do so by using the louvain method for graph partitioning.

In [ ]:
#| export
@patch
def get_coalitions(self: SFMTL, G):
    correct_clients_indices = nx.get_node_attributes(G, 'label')
    partitions = community_louvain.best_partition(G)
    communities = defaultdict(list)
    for client, community in partitions.items():
        communities[community].append(client)
    communities = dict(communities)

    for community, clients in communities.items():
        communities[community] = [correct_clients_indices[client] for client in clients]

    return communities


### Aggregation

The aggregation rule here is two-folds:
- Representations are aggregated as:
  $$ h_c = \sum_{k \in C_{j}} \frac{\zeta_k}{\sum_{k \in C_{j}} \zeta_k}h_k$$
  where $\zeta$ is the shapely value.
  
The Classification heads are aggregated as:
$$  w_k^{(t+1)} = w_{k, R}^{(t)} - \lambda \eta_2 \sum_{\ell \in C_{j}} a_{k, \ell} (w_{k,R}^{(t)} - w_{\ell, R}^{(t)})$$

In [ ]:
#| export
@patch
def aggregate(self: SFMTL, lst_active_ids, comm_round, len_clients_ds):

    self.graph, self.akl_connection = self.build_random_graph(lst_active_ids, comm_round)
    graph_path = os.path.join(self.cfg.save_dir, str(comm_round), f"graph_{str(comm_round)}.gpickle")
    with open(graph_path, "wb") as f:
        pickle.dump(self.graph, f, pickle.HIGHEST_PROTOCOL)

    self.coalitions = self.get_coalitions(self.graph)
    coalitions_path = os.path.join(self.cfg.save_dir, str(comm_round), "coalitions.pth")
    torch.save(self.coalitions, coalitions_path)

    global_lr = float(self.cfg.optimizer.lr) * float(self.cfg.local_epochs)
    reg_param = self.cfg.lambda_
    
    with torch.no_grad():
        coalitions_reprs = {}
        for col_ind, lst_clients in self.coalitions.items():

            m_t = sum(len_clients_ds[id] for id in lst_clients)
            for i, id in enumerate(lst_clients):
                if not id in lst_active_ids:
                    continue
                state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
                state = torch.load(state_path, weights_only= False)
                client_h = state['h']

                if i == 0:
                    col_repr = torch.zeros_like(client_h)

                n_k = len_clients_ds[id]
                weight =  n_k / m_t 

                col_repr.add_(weight * client_h)
            coalitions_reprs[col_ind] = col_repr
            

        for col_ind, lst_clients in self.coalitions.items():
            for i, id in enumerate(lst_clients):
                if not id in lst_active_ids:
                    continue
                state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{id}", "state.pth")
                
                state = torch.load(state_path, weights_only= False)
                client_model = state['model']

                client_diff = {
                    key: torch.zeros_like(value) 
                    for key, value in client_model.items() if key.startswith("fc2") or key.startswith("dropout")
                }

                for j, other_id in enumerate(lst_clients):
                    if i == j:
                        continue
                    other_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"local_output_{other_id}", "state.pth")
                    
                    other_state = torch.load(other_state_path, weights_only= False)
                    other_client_model = other_state['model']

                    a_kl = self.akl_connection[i, j]
                    for key in client_model.keys():
                        if key.startswith("fc2") or key.startswith("dropout"):
                            client_diff[key].add_(a_kl * (client_model[key] - other_client_model[key]))

                for key in client_model.keys():
                    if key.startswith("fc2") or key.startswith("dropout"):
                        client_model[key].sub_(global_lr * reg_param * client_diff[key])

                clinet_state = {
                    'model': client_model,
                    'h': state['h'],
                    'h_c': coalitions_reprs[col_ind],
                }

                agg_client_state_path = os.path.join(self.cfg.save_dir, str(comm_round), f"aggregated_model_{id}", "state.pth")
                
                if not os.path.exists(os.path.dirname(agg_client_state_path)):
                    os.makedirs(os.path.dirname(agg_client_state_path))

                torch.save(clinet_state, agg_client_state_path)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()